Large language models can answer questions fluently, but they often produce answers that are not grounded in a specific document. Retrieval-Augmented Generation (RAG) addresses this by first retrieving the most relevant passages from a source document and then asking a language model to answer using only that material. This notebook implements a complete RAG pipeline that lets a user upload a PDF and ask questions about it in natural language.

The system extracts text from the PDF page by page with PyMuPDF, splits it into overlapping 150-word chunks, and converts each chunk into a vector using the all-MiniLM-L6-v2 sentence embedding model. These vectors are stored in a FAISS index for fast similarity search. When a question is asked, it is embedded in the same way, and the three closest chunks are retrieved and passed to google/flan-t5-large with an instruction to answer only from the given context. The answer is returned together with the source page numbers, so every response can be checked against the original document. The pipeline is built entirely from open-source components, so it needs no paid API or external service.

Installing Libraries

In [1]:
!pip install -q pymupdf sentence-transformers faiss-cpu transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 52.0 MB/s eta 0:00:00


Uploading PDF

In [2]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print("Loaded:", pdf_path)

Saving AI_MS_module1.pdf to AI_MS_module1.pdf
Loaded: AI_MS_module1.pdf


PyMuPDF

In [3]:
import fitz  # PyMuPDF

doc = fitz.open(pdf_path)
pages = []
for i, page in enumerate(doc):
    text = page.get_text().strip()
    if text:
        pages.append({"page": i + 1, "text": text})

print("Pages with text:", len(pages))

Pages with text: 73


Chunking

In [4]:
def chunk_text(text, chunk_size=150, overlap=30):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks

chunks = []
for p in pages:
    for c in chunk_text(p["text"]):
        chunks.append({"page": p["page"], "text": c})

print("Total chunks:", len(chunks))

Total chunks: 81


MiniLM embeddings - (all-MiniLM-L6-v2)



In [5]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [c["text"] for c in chunks]
embeddings = embedder.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
print("Embeddings shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Embeddings shape: (81, 384)


FAISS index (vector database)

In [6]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)

print("Vectors in index:", index.ntotal)

Vectors in index: 81


Loading LLM

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
llm_model = llm_model.to(device)

def llm(prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(device)
    output_ids = llm_model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.13GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Question → embedding → FAISS search → LLM → answer

In [8]:
def ask(question, top_k=3):
    # Question to vector
    q_vec = embedder.encode([question], convert_to_numpy=True, normalize_embeddings=True)

    # FAISS search: top-k similar chunks
    scores, ids = index.search(q_vec, top_k)
    retrieved = [chunks[i] for i in ids[0]]

    # Build prompt: question + retrieved context
    context = "\n\n".join(r["text"] for r in retrieved)
    prompt = (
        "Answer the question using only the context below. "
        "If the answer is not in the context, say you don't know.\n\n"
        f"Question: {question}\n\n"
        f"Context:\n{context}\n\n"
        "Answer:"
    )

    # LLM generates the answer
    answer = llm(prompt)

    pages_used = sorted(set(r["page"] for r in retrieved))
    return answer, pages_used

Chat Loop

In [9]:
while True:
    q = input("\nAsk a question (type 'exit' to stop): ")
    if q.lower() == "exit":
        break
    answer, pages_used = ask(q)
    print("Answer:", answer)
    print("Source pages:", pages_used)


Ask a question (type 'exit' to stop): is SMILES in there
Answer: Molecular Representations  SMILES, InChI - string-based formats
Source pages: [40, 41, 63]

Ask a question (type 'exit' to stop): what is written about fingerprints
Answer: fingerprints
Source pages: [48, 50, 53]

Ask a question (type 'exit' to stop): stop
Answer: not enough information
Source pages: [20, 22, 76]

Ask a question (type 'exit' to stop): exit


This project shows that a working document question-answering system can be built with a small set of open-source tools. Chunking, semantic embedding, vector search and a lightweight generative model were enough to give accurate, concise answers to questions about a technical PDF, such as the full forms of SMILES and InChI and the main CNN variants. Returning the source pages for each answer adds transparency and makes the output easy to verify.

The system also has limitations. Flan-T5 produces short answers and handles about 1024 tokens of input, so it is better suited to factual lookups and definitions than to detailed explanations. Fixed-size chunking can split related ideas across chunks, and retrieval quality depends on the embedding model. Future work could use a stronger LLM, semantic or section-aware chunking, a reranking step, and a quantitative evaluation of retrieval and answer quality. Even in its current form, the notebook is a clear and practical demonstration of how RAG makes language models more reliable on domain-specific documents.